# 01 - Análise Exploratória e Modelagem de Propensão (SmartRetail)

Este notebook documenta a fase experimental de Ciência de Dados para a **SmartRetail**. Desenvolvemos aqui o pipeline para analisar dados históricos de clientes e treinar um classificador supervisionado de propensão de resposta a campanhas de marketing digital.

### Estrutura do Projeto Esperada:
O arquivo contendo as informações deve estar localizado em:
`data/clientes_marketing.csv`

## 1. Importação de Bibliotecas

Carregamos as dependências do ecossistema Python necessárias para visualização, manipulação e treinamento de modelos.

In [ ]:
import logging
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Definições estéticas para gráficos do Seaborn
sns.set_theme(style="whitegrid")

## 2. Carregamento dos Dados da SmartRetail

Realizamos a leitura do arquivo `clientes_marketing.csv` armazenado na subpasta isolada `data` e realizamos uma inspeção visual inicial de sua estrutura.

In [ ]:
caminho_dados = Path("data") / "clientes_marketing.csv"

# Verificação estrutural do arquivo
if not caminho_dados.exists():
    logging.error(f"Erro de Execução: Dataset não localizado em {caminho_dados.resolve()}")
else:
    df = pd.read_csv(caminho_dados)
    print(f"Dataset SmartRetail carregado. Linhas: {df.shape[0]} | Colunas: {df.shape[1]}\n")
    display(df.head())

In [ ]:
# Avaliação estrutural dos tipos de dados e valores ausentes (nulos)
df.info()

## 3. Divisão do Conjunto de Dados

Dividimos o dataset na proporção de **80% para treinamento** e **20% para teste**, aplicando estratificação sobre a variável alvo `RespondeuCampanha` para manter o balanceamento original em ambas as amostras.

In [ ]:
X = df.drop(columns=['RespondeuCampanha'])
y = df['RespondeuCampanha']

# Amostragem estratificada
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Conjunto de Treinamento: {X_train.shape[0]} observações")
print(f"Conjunto de Teste (Validação): {X_test.shape[0]} observações")

## 4. Ajuste do Classificador (Decision Tree)

Treinamos a árvore de decisão definindo `max_depth=3` para limitar sua profundidade lúdica e mitigar chances de sobreajuste aos dados locais de treinamento.

In [ ]:
modelo = DecisionTreeClassifier(random_state=42, max_depth=3)
modelo.fit(X_train, y_train)

print("Ajuste do modelo de propensão SmartRetail finalizado.")

## 5. Avaliação Estatística de Desempenho

Aplicamos o modelo de propensão ao conjunto isolado de testes e analisamos os resultados obtidos de forma consolidada e detalhada.

In [ ]:
# Predições
y_pred = modelo.predict(X_test)
acuracia = accuracy_score(y_test, y_pred)

print(f"Acurácia calculada em teste: {acuracia * 100:.2f}%\n")

# Gráfico da Matriz de Confusão
matriz = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(
    matriz, 
    annot=True, 
    fmt='d', 
    cmap='Blues', 
    xticklabels=['Não Respondeu (0)', 'Respondeu (1)'], 
    yticklabels=['Não Respondeu (0)', 'Respondeu (1)']
)
plt.title('Matriz de Confusão - SmartRetail')
plt.ylabel('Rótulo Real')
plt.xlabel('Predição do Classificador')
plt.show()

# Relatório Estatístico Detalhado
print("\nRelatório de Classificação Detalhado:")
print(classification_report(y_test, y_pred))

## 6. Visualização do Fluxo de Decisões do Modelo

Abaixo, renderizamos o diagrama das regras de ramificação da árvore de propensão. Este fluxo descreve visualmente os critérios de filtragem de perfil que serão passados para a equipe de CRM e Negócios.

In [ ]:
plt.figure(figsize=(14, 8))
plot_tree(
    modelo, 
    feature_names=X.columns, 
    class_names=['Não Respondeu (0)', 'Respondeu (1)'], 
    filled=True, 
    rounded=True, 
    fontsize=10
)
plt.title("Visualização das Decisões de Classificação - SmartRetail", fontsize=14)
plt.show()